# Kết quả định tính Dense Mapped Logit-KD trên MS COCO

Notebook chỉ thực hiện **suy luận định tính** cho mô hình T1–S1 (ExpansionNet v2 → MobileNetV3-Large, Dense Mapped Logit-KD, seed 123, Dynamic Weight INT8).

Đầu ra gồm:

- một ảnh tổng hợp chứa từ **6 đến 8 mẫu**, bố trí thành **hai cột bằng nhau**;
- mỗi mẫu được trình bày theo đúng thứ tự: **ảnh → image ID → mã (i), (ii), ... → Reference → Generated**;
- một bản PDF cùng bố cục để chèn vào Word mà không giảm chất lượng;
- một tệp JSON lưu image ID, caption tham chiếu, caption sinh, độ trễ và SHA-256 của mô hình.

Hình tổng hợp không có tiêu đề chung hoặc tiêu đề riêng cho từng ảnh. Toàn bộ phần chữ dùng Times New Roman; nếu môi trường Kaggle không có font này, notebook tự chuyển sang Times hoặc DejaVu Serif.

## Kaggle Input cần gắn

1. Output INT8 Validation Preflight chứa encoder/decoder INT8 và vocabulary;
2. Output notebook MS COCO Data Contract chứa `mscoco_validation.parquet` hoặc `dataset_coco.json`;
3. Dataset `mscoco2014-1` chứa ảnh `val2014`;
4. Dataset ONNX Runtime offline, chỉ cần khi Kaggle chưa cài ONNX Runtime tương thích.

Cấu hình Kaggle: **Accelerator None/CPU**, **Internet Off**. Notebook không huấn luyện, không chọn lại checkpoint và không đọc Held-out Test.


## Cell 1 — Cấu hình và ONNX Runtime

Có thể điền `IMAGE_IDS` bằng 6–8 MS COCO image ID. Nếu để `None`, notebook lấy 8 ảnh Validation tại các vị trí phân bố đều trong `QUALITATIVE_INDICES`, theo thứ tự image ID tăng dần.


In [ ]:
import hashlib
import importlib
import json
import math
import subprocess
import sys
import textwrap
import time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
assert INPUT_ROOT.is_dir(), 'Notebook phải chạy trên Kaggle hoặc môi trường có /kaggle/input.'

# -------------------------- CẤU HÌNH NGƯỜI DÙNG --------------------------
# Có thể nhập 6–8 image ID cụ thể. None = dùng các vị trí Validation bên dưới.
IMAGE_IDS = None
# Mặc định 8 ảnh, phân bố đều trên 5.000 ảnh Karpathy Validation.
QUALITATIVE_INDICES = (0, 714, 1428, 2142, 2856, 3570, 4284, 4999)
REFERENCE_INDEX = 0              # Chọn 1 caption tham chiếu cho mỗi ảnh
MINIMUM_IMAGES = 6
MAXIMUM_IMAGES = 8
FIGURE_DPI = 300
# -----------------------------------------------------------------------

SPLIT = 'val'
SEED = 123
BEAM_SIZE = 3
LENGTH_PENALTY = 0.7
MAXIMUM_STEPS = 31
MINIMUM_LEXICAL_TOKENS = 2
PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3

EXPECTED = {
    'encoder_sha256': 'b015b503e8af8e9d86db04cbb00bd7a1f80d92e3274c590f84cfedf7f237efd4',
    'decoder_sha256': 'ee7a324290aeae95613b5de1f8bcb88f1e88bf1652c98a9fe55389412c25d2cd',
    'vocabulary_sha256': '1c6c7e6146d2700932c82be6f076d93153c757c10a1844076abaea9c11496f97',
}

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def ensure_onnxruntime():
    try:
        import onnxruntime as runtime
        return runtime, 'preinstalled'
    except Exception as first_error:
        python_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
        wheels = sorted(INPUT_ROOT.rglob(f'onnxruntime-*-{python_tag}-{python_tag}-*.whl'))
        if not wheels:
            raise RuntimeError(
                'Không import được onnxruntime và không tìm thấy wheel tương thích. '
                'Hãy gắn input ONNX_Runtime_Python313_Offline.'
            ) from first_error
        wheel_dirs = sorted({str(path.parent) for path in wheels})
        command = [sys.executable, '-m', 'pip', 'install', '--no-index']
        for directory in wheel_dirs:
            command.extend(['--find-links', directory])
        command.append('onnxruntime')
        completed = subprocess.run(command, text=True, capture_output=True)
        print(completed.stdout)
        if completed.returncode != 0:
            print(completed.stderr)
            raise RuntimeError('Cài ONNX Runtime offline thất bại.')
        importlib.invalidate_caches()
        import onnxruntime as runtime
        return runtime, str(wheels[-1])

ort, ORT_SOURCE = ensure_onnxruntime()
assert 'CPUExecutionProvider' in ort.get_available_providers()
print('ONNX Runtime :', ort.__version__)
print('ORT source   :', ORT_SOURCE)
print('Data split   : Karpathy Validation (Held-out Test không được đọc)')

## Cell 2 — Khám phá model, vocabulary và dữ liệu Validation

Notebook ưu tiên nhận diện model bằng SHA-256 đã khóa. Vì vậy việc đổi tên thư mục Kaggle Input không ảnh hưởng kết quả.

In [ ]:
def locate_artifact(role, expected_sha256, suffix, name_terms):
    candidates = []
    for path in INPUT_ROOT.rglob(f'*{suffix}'):
        lowered = path.name.lower()
        if all(term in lowered for term in name_terms):
            candidates.append(path.resolve())
    matches = [path for path in candidates if sha256(path) == expected_sha256]
    # Output preflight lưu cùng artifact ở package/ và candidates/. Bản package/
    # là đầu vào triển khai chính thức nên được ưu tiên khi SHA-256 trùng nhau.
    package_matches = [path for path in matches if 'package' in {part.lower() for part in path.parts}]
    if len(package_matches) == 1:
        return package_matches[0]
    if len(matches) != 1:
        details = '\n'.join(f'  - {path}' for path in candidates[:30]) or '  (không có)'
        raise AssertionError(
            f'Không tìm thấy đúng một {role} có SHA-256 đã khóa. Candidates:\n{details}'
        )
    return matches[0]

ENCODER_PATH = locate_artifact(
    'encoder INT8', EXPECTED['encoder_sha256'], '.onnx', ('encoder', 'int8')
)
DECODER_PATH = locate_artifact(
    'decoder INT8', EXPECTED['decoder_sha256'], '.onnx', ('decoder', 'int8')
)

vocab_candidates = sorted({
    path.resolve()
    for pattern in ('*vocabulary*.json', '*vocab*.json')
    for path in INPUT_ROOT.rglob(pattern)
})
VOCAB_MATCHES = [
    path for path in vocab_candidates
    if sha256(path) == EXPECTED['vocabulary_sha256']
]
PACKAGE_VOCAB_MATCHES = [
    path for path in VOCAB_MATCHES
    if 'package' in {part.lower() for part in path.parts}
]
if len(PACKAGE_VOCAB_MATCHES) == 1:
    VOCAB_PATH = PACKAGE_VOCAB_MATCHES[0]
elif len(VOCAB_MATCHES) == 1:
    VOCAB_PATH = VOCAB_MATCHES[0]
else:
    raise AssertionError((
        'Không xác định được vocabulary chính thức. Hãy giữ output INT8 Validation Preflight.',
        VOCAB_MATCHES,
    ))

with VOCAB_PATH.open('r', encoding='utf-8') as handle:
    vocabulary = json.load(handle)
ID_TO_TOKEN = list(vocabulary['id_to_token'])
assert len(ID_TO_TOKEN) == 10080
assert ID_TO_TOKEN[:4] == ['<pad>', '<bos>', '<eos>', '<unk>']

REFERENCES = defaultdict(list)
FILENAMES = {}
parquet_matches = sorted(INPUT_ROOT.rglob('mscoco_validation.parquet'))
json_matches = sorted(INPUT_ROOT.rglob('dataset_coco.json'))

if len(parquet_matches) == 1:
    import pyarrow.parquet as pq
    contract_path = parquet_matches[0]
    table = pq.read_table(
        contract_path,
        columns=['annotation_id', 'image_id', 'karpathy_split', 'source_partition', 'caption'],
    )
    rows = sorted(table.to_pylist(), key=lambda row: (int(row['image_id']), int(row['annotation_id'])))
    for row in rows:
        if row['karpathy_split'] != 'val':
            continue
        assert row['source_partition'] == 'val2014'
        image_id = int(row['image_id'])
        REFERENCES[image_id].append(str(row['caption']).strip())
        FILENAMES[image_id] = f'COCO_val2014_{image_id:012d}.jpg'
elif len(json_matches) == 1:
    contract_path = json_matches[0]
    with contract_path.open('r', encoding='utf-8') as handle:
        karpathy = json.load(handle)
    for item in karpathy['images']:
        if item.get('split') != 'val':
            continue
        image_id = int(item.get('cocoid', item.get('imgid')))
        FILENAMES[image_id] = item['filename']
        REFERENCES[image_id] = [
            str(sentence.get('raw', '')).strip()
            for sentence in item.get('sentences', [])
            if str(sentence.get('raw', '')).strip()
        ]
else:
    raise AssertionError(
        'Cần đúng một mscoco_validation.parquet hoặc dataset_coco.json từ Data Contract. '
        f'Parquet={parquet_matches}; JSON={json_matches}'
    )

VALIDATION_IMAGE_IDS = tuple(sorted(REFERENCES))
assert len(VALIDATION_IMAGE_IDS) == 5000, len(VALIDATION_IMAGE_IDS)
assert sum(len(values) for values in REFERENCES.values()) == 25010
assert all(REFERENCES[image_id] for image_id in VALIDATION_IMAGE_IDS)
assert SPLIT == 'val', 'Notebook định tính này không được chuyển sang Test.'

if IMAGE_IDS is None:
    assert MINIMUM_IMAGES <= len(QUALITATIVE_INDICES) <= MAXIMUM_IMAGES, (
        'QUALITATIVE_INDICES phải chứa từ 6 đến 8 vị trí.'
    )
    assert len(set(QUALITATIVE_INDICES)) == len(QUALITATIVE_INDICES)
    assert all(0 <= index < len(VALIDATION_IMAGE_IDS) for index in QUALITATIVE_INDICES)
    SELECTED_IMAGE_IDS = tuple(VALIDATION_IMAGE_IDS[index] for index in QUALITATIVE_INDICES)
else:
    assert MINIMUM_IMAGES <= len(IMAGE_IDS) <= MAXIMUM_IMAGES, (
        'IMAGE_IDS phải chứa từ 6 đến 8 image ID.'
    )
    SELECTED_IMAGE_IDS = tuple(int(image_id) for image_id in IMAGE_IDS)
    assert len(set(SELECTED_IMAGE_IDS)) == len(SELECTED_IMAGE_IDS), (
        'Các image ID phải khác nhau.'
    )
    assert all(image_id in REFERENCES for image_id in SELECTED_IMAGE_IDS), (
        'Có IMAGE_ID không thuộc Karpathy Validation.', SELECTED_IMAGE_IDS
    )

SELECTED_ITEMS = []
for image_id in SELECTED_IMAGE_IDS:
    selected_filename = FILENAMES[image_id]
    image_matches = sorted({path.resolve() for path in INPUT_ROOT.rglob(selected_filename)})
    assert image_matches, f'Không tìm thấy ảnh {selected_filename}; hãy gắn dataset mscoco2014-1.'
    if len(image_matches) > 1:
        preferred = [path for path in image_matches if 'val2014' in str(path).lower()]
        image_matches = preferred or image_matches
    assert 0 <= REFERENCE_INDEX < len(REFERENCES[image_id])
    SELECTED_ITEMS.append({
        'image_id': image_id,
        'filename': selected_filename,
        'image_path': image_matches[0],
        'reference_caption': REFERENCES[image_id][REFERENCE_INDEX],
    })

print('Encoder       :', ENCODER_PATH)
print('Decoder       :', DECODER_PATH)
print('Vocabulary    :', VOCAB_PATH)
print('Data contract :', contract_path)
print('Selected      :', SELECTED_IMAGE_IDS)
for ordinal, item in enumerate(SELECTED_ITEMS, start=1):
    print(f"Image {ordinal}       : id={item['image_id']} | {item['image_path']}")
    print(f"References {ordinal}  : {len(REFERENCES[item['image_id']])}")

## Cell 3 — Tiền xử lý và Beam Search

Cấu hình được giữ đúng với gói triển khai Android: resize cạnh ngắn 256, center crop 224, chuẩn hóa ImageNet, Beam-3, length penalty 0,7, tối đa 31 bước.

In [ ]:
IMAGENET_MEAN = np.asarray([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.asarray([0.229, 0.224, 0.225], dtype=np.float32)

def preprocess_image(image):
    image = image.convert('RGB')
    width, height = image.size
    if width <= height:
        resized_width = 256
        resized_height = int(256 * height / width)
    else:
        resized_height = 256
        resized_width = int(256 * width / height)
    image = image.resize((resized_width, resized_height), Image.Resampling.BICUBIC)
    left = int(round((resized_width - 224) / 2.0))
    top = int(round((resized_height - 224) / 2.0))
    image = image.crop((left, top, left + 224, top + 224))
    array = np.asarray(image, dtype=np.float32) / 255.0
    array = (array - IMAGENET_MEAN) / IMAGENET_STD
    return np.ascontiguousarray(array.transpose(2, 0, 1)[None, ...], dtype=np.float32)

session_options = ort.SessionOptions()
session_options.intra_op_num_threads = 1
session_options.inter_op_num_threads = 1
session_options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

ENCODER = ort.InferenceSession(
    str(ENCODER_PATH), sess_options=session_options, providers=['CPUExecutionProvider']
)
DECODER = ort.InferenceSession(
    str(DECODER_PATH), sess_options=session_options, providers=['CPUExecutionProvider']
)
assert [value.name for value in ENCODER.get_inputs()] == ['image']
assert [value.name for value in ENCODER.get_outputs()] == ['visual_memory']
assert [value.name for value in DECODER.get_inputs()] == [
    'decoder_input_ids', 'visual_memory', 'decoder_attention_mask'
]
assert [value.name for value in DECODER.get_outputs()] == ['logits']

def decode_tokens(token_ids):
    words = []
    for token_id in token_ids:
        token_id = int(token_id)
        if token_id == EOS_ID:
            break
        if token_id in {PAD_ID, BOS_ID, UNK_ID}:
            continue
        words.append(ID_TO_TOKEN[token_id])
    return ' '.join(words)

def rank_beam(beam):
    tokens, raw_score, _ = beam
    length = max(1, len(tokens) - 1)
    return raw_score / (length ** LENGTH_PENALTY)

def beam_generate(image_tensor):
    started = time.perf_counter()
    memory = np.asarray(
        ENCODER.run(['visual_memory'], {'image': image_tensor})[0], dtype=np.float32
    )
    assert memory.shape == (1, 49, 256), memory.shape
    beams = [([BOS_ID], 0.0, False)]

    for _ in range(MAXIMUM_STEPS):
        active = [beam for beam in beams if not beam[2]]
        if not active:
            break

        decoder_ids = np.zeros((BEAM_SIZE, 31), dtype=np.int64)
        decoder_mask = np.zeros((BEAM_SIZE, 31), dtype=np.int64)
        expanded_memory = np.repeat(memory, BEAM_SIZE, axis=0)
        for row_index, (tokens, _, _) in enumerate(active):
            length = min(len(tokens), 31)
            decoder_ids[row_index, :length] = np.asarray(tokens[:length], dtype=np.int64)
            decoder_mask[row_index, :length] = 1

        logits = DECODER.run(
            ['logits'],
            {
                'decoder_input_ids': decoder_ids,
                'visual_memory': np.ascontiguousarray(expanded_memory),
                'decoder_attention_mask': decoder_mask,
            },
        )[0]
        assert logits.shape == (3, 31, 10080), logits.shape

        candidates = [beam for beam in beams if beam[2]]
        for row_index, (tokens, score, _) in enumerate(active):
            position = min(len(tokens) - 1, 30)
            values = logits[row_index, position].astype(np.float64)
            values = values - np.max(values)
            log_probs = values - np.log(np.exp(values).sum())
            log_probs[[PAD_ID, BOS_ID, UNK_ID]] = -np.inf
            lexical_count = max(0, len(tokens) - 1)
            if lexical_count < MINIMUM_LEXICAL_TOKENS:
                log_probs[EOS_ID] = -np.inf
            top_ids = np.argpartition(log_probs, -BEAM_SIZE)[-BEAM_SIZE:]
            top_ids = top_ids[np.argsort(log_probs[top_ids])[::-1]]
            for token_id in top_ids.tolist():
                candidates.append((
                    tokens + [int(token_id)],
                    float(score + log_probs[token_id]),
                    int(token_id) == EOS_ID,
                ))
        beams = sorted(candidates, key=rank_beam, reverse=True)[:BEAM_SIZE]
        if all(beam[2] for beam in beams):
            break

    best = max(beams, key=rank_beam)
    latency_ms = (time.perf_counter() - started) * 1000.0
    return {
        'token_ids': [int(value) for value in best[0]],
        'caption': decode_tokens(best[0]),
        'raw_beam_score': float(best[1]),
        'normalized_beam_score': float(rank_beam(best)),
        'latency_ms': float(latency_ms),
    }

print('Encoder/decoder ONNX loaded successfully.')
print('Decoding: Beam-3 | length penalty=0.7 | maximum steps=31')

## Cell 4 — Sinh caption và xuất hình tổng hợp hai cột

Mỗi mẫu được trình bày theo thứ tự: ảnh, image ID, mã La Mã, caption tham chiếu và caption do mô hình sinh. Không dùng tiêu đề để dành tối đa diện tích cho ảnh và phần chữ.


In [ ]:
OUTPUT_JSON = WORK_ROOT / 'Dense_Mapped_LogitKD_Qualitative_MSCOCO.json'
OUTPUT_PNG = WORK_ROOT / 'Dense_Mapped_LogitKD_Qualitative_MSCOCO_2col.png'
OUTPUT_PDF = WORK_ROOT / 'Dense_Mapped_LogitKD_Qualitative_MSCOCO_2col.pdf'
RESULT_ITEMS = []
DISPLAY_IMAGES = []
ROMAN_LABELS = ('i', 'ii', 'iii', 'iv', 'v', 'vi', 'vii', 'viii')

# Times New Roman được ưu tiên; Times và DejaVu Serif là font dự phòng trên Kaggle.
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size': 24,
    'axes.unicode_minus': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

for ordinal, item in enumerate(SELECTED_ITEMS, start=1):
    with Image.open(item['image_path']) as image_handle:
        display_image = image_handle.convert('RGB').copy()
    image_tensor = preprocess_image(display_image)
    prediction = beam_generate(image_tensor)
    assert prediction['caption'].strip(), f"Caption ảnh {item['image_id']} đang rỗng."

    DISPLAY_IMAGES.append(display_image)
    RESULT_ITEMS.append({
        'ordinal': ordinal,
        'marker': f"({ROMAN_LABELS[ordinal - 1]})",
        'image_id': item['image_id'],
        'filename': item['filename'],
        'reference_index': REFERENCE_INDEX,
        'reference_caption': item['reference_caption'],
        'all_reference_captions': REFERENCES[item['image_id']],
        'prediction': prediction,
    })

assert MINIMUM_IMAGES <= len(RESULT_ITEMS) <= MAXIMUM_IMAGES
assert len(RESULT_ITEMS) == len(DISPLAY_IMAGES)

column_count = 2
row_count = math.ceil(len(RESULT_ITEMS) / column_count)
# Khổ hình gần tỷ lệ vùng in A4 trong Word (6,5 x 9,25 inch).
# Khi Word thu hình từ 13,2 inch xuống 6,5 inch, chữ 24–28 pt còn
# khoảng 12–14 pt trên trang in, đủ rõ để đọc mà không cần phóng to.
figure = plt.figure(
    figsize=(13.2, 4.7 * row_count),
    facecolor='white',
    constrained_layout=False,
)
outer_grid = figure.add_gridspec(
    row_count,
    column_count,
    left=0.035,
    right=0.965,
    bottom=0.025,
    top=0.985,
    wspace=0.10,
    hspace=0.14,
)

for index, (item, display_image) in enumerate(zip(RESULT_ITEMS, DISPLAY_IMAGES)):
    row_index, column_index = divmod(index, column_count)
    panel_grid = outer_grid[row_index, column_index].subgridspec(
        2, 1, height_ratios=(2.4, 3.2), hspace=0.020
    )
    image_axis = figure.add_subplot(panel_grid[0])
    text_axis = figure.add_subplot(panel_grid[1])

    image_axis.imshow(display_image)
    image_axis.axis('off')

    text_axis.axis('off')
    text_axis.set_xlim(0, 1)
    text_axis.set_ylim(0, 1)
    text_axis.text(
        0.5,
        0.92,
        f"image_id: {item['image_id']}",
        fontsize=28,
        fontweight='bold',
        ha='center',
        va='top',
        color='#111111',
    )
    text_axis.text(
        0.5,
        0.75,
        item['marker'],
        fontsize=27,
        fontweight='bold',
        ha='center',
        va='top',
        color='#111111',
    )
    reference_text = textwrap.fill(f"Reference: {item['reference_caption']}", width=35)
    generated_text = textwrap.fill(f"Generated: {item['prediction']['caption']}", width=35)
    text_axis.text(
        0.02,
        0.58,
        reference_text,
        fontsize=22,
        ha='left',
        va='top',
        color='#111111',
        linespacing=1.12,
    )
    text_axis.text(
        0.02,
        0.20,
        generated_text,
        fontsize=22,
        ha='left',
        va='top',
        color='#111111',
        linespacing=1.12,
    )

# Nếu số ảnh lẻ, ẩn ô còn trống để hai cột vẫn có cùng chiều rộng.
for empty_index in range(len(RESULT_ITEMS), row_count * column_count):
    row_index, column_index = divmod(empty_index, column_count)
    empty_axis = figure.add_subplot(outer_grid[row_index, column_index])
    empty_axis.axis('off')

# Không đặt suptitle hoặc title theo yêu cầu trình bày báo cáo.
figure.savefig(
    OUTPUT_PNG,
    dpi=FIGURE_DPI,
    bbox_inches='tight',
    pad_inches=0.18,
    facecolor='white',
)
figure.savefig(
    OUTPUT_PDF,
    bbox_inches='tight',
    pad_inches=0.18,
    facecolor='white',
)
plt.show()
plt.close(figure)

assert OUTPUT_PNG.is_file() and OUTPUT_PNG.stat().st_size > 0
assert OUTPUT_PDF.is_file() and OUTPUT_PDF.stat().st_size > 0

result = {
    'schema_version': 'dense_mapped_logitkd_qualitative_mscoco_v2',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model': {
        'pair': 'T1-S1',
        'teacher': 'ExpansionNet v2',
        'student': 'MobileNetV3-Large',
        'method': 'Dense Mapped Logit-KD',
        'seed': SEED,
        'precision': 'Dynamic Weight INT8',
        'encoder_sha256': sha256(ENCODER_PATH),
        'decoder_sha256': sha256(DECODER_PATH),
        'vocabulary_sha256': sha256(VOCAB_PATH),
    },
    'data': {
        'dataset': 'MS COCO 2014',
        'split': 'Karpathy Validation',
        'number_of_images': len(RESULT_ITEMS),
        'held_out_test_opened': False,
    },
    'layout': {
        'columns': column_count,
        'rows': row_count,
        'font_family': 'Times New Roman',
        'font_fallback': ['Times', 'DejaVu Serif'],
        'body_font_size_pt': 22,
        'identifier_font_size_pt': 28,
        'word_effective_body_font_size_pt_at_6_5_in_width': 10.8,
        'word_effective_identifier_font_size_pt_at_6_5_in_width': 13.8,
        'dpi': FIGURE_DPI,
        'title_used': False,
        'output_png': str(OUTPUT_PNG),
        'output_pdf': str(OUTPUT_PDF),
    },
    'decoding': {
        'beam_size': BEAM_SIZE,
        'length_penalty': LENGTH_PENALTY,
        'maximum_steps': MAXIMUM_STEPS,
    },
    'results': RESULT_ITEMS,
}
with OUTPUT_JSON.open('w', encoding='utf-8') as handle:
    json.dump(result, handle, ensure_ascii=False, indent=2)

print(f"\nKẾT QUẢ ĐỊNH TÍNH — {len(RESULT_ITEMS)} ẢNH — 2 CỘT")
for item in RESULT_ITEMS:
    print(f"\n{item['marker']} image_id={item['image_id']}")
    print('Reference :', item['reference_caption'])
    print('Generated :', item['prediction']['caption'])
    print('Latency   :', f"{item['prediction']['latency_ms']:.1f} ms")
print('\nPNG  :', OUTPUT_PNG)
print('PDF  :', OUTPUT_PDF)
print('JSON :', OUTPUT_JSON)


## Output kỳ vọng

- `/kaggle/working/Dense_Mapped_LogitKD_Qualitative_MSCOCO_2col.png`
- `/kaggle/working/Dense_Mapped_LogitKD_Qualitative_MSCOCO_2col.pdf`
- `/kaggle/working/Dense_Mapped_LogitKD_Qualitative_MSCOCO.json`

PNG và PDF đều chứa 6–8 ảnh trong hai cột bằng nhau. Mỗi mẫu theo thứ tự: ảnh → image ID → mã La Mã → Reference → Generated. Hình không có tiêu đề; chữ dùng Times New Roman cỡ nguồn 22–28 pt, tương đương khoảng 11–14 pt khi đặt vừa chiều rộng vùng in 6,5 inch trong Word.
